# 01 — Explore the raw data

First pass over the three raw datasets. Goal here is **not** to clean anything, just to:

- confirm each file loads (especially the PPR, which is cp1252-encoded)
- check shape, dtypes, and where the nulls are
- learn what values the key dimensions actually take, so the cleaning notebook (`02_clean.ipynb`) has a plan

The quirks called out in `data/README.md` are verified against the real files below.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RAW = Path("..") / "data" / "raw"
rtb_path = RAW / "rtb_rent_index.csv"
earnings_path = RAW / "cso_earnings.csv"
ppr_path = RAW / "ppr.csv"

for p in (rtb_path, earnings_path, ppr_path):
    print(f"{p.name:24s} {'OK' if p.exists() else 'MISSING':8s} {p.stat().st_size/1e6:.1f} MB")

rtb_rent_index.csv       OK       92.6 MB
cso_earnings.csv         OK       0.1 MB
ppr.csv                  OK       107.1 MB


## 1. RTB Rent Index

Quarterly average rents by location, property type, and bedroom count. Expect ~880k rows and a lot of null `VALUE` where there weren't enough registered tenancies to publish a number.

In [2]:
rtb = pd.read_csv(rtb_path)
print(rtb.shape)
rtb.head()

(880404, 7)


,STATISTIC Label,Quarter,Number of Bedrooms,Property Type,Location,UNIT,VALUE
0,RTB Average Monthly Rent Report,2014Q1,All bedrooms,All property types,Carlow,Euro,579.65
1,RTB Average Monthly Rent Report,2014Q1,All bedrooms,All property types,Carlow Town,Euro,595.60
2,RTB Average Monthly Rent Report,2014Q1,All bedrooms,All property types,"Graiguecullen, Carlow",Euro,545.84
3,RTB Average Monthly Rent Report,2014Q1,All bedrooms,All property types,"Tullow, Carlow",Euro,548.46
4,RTB Average Monthly Rent Report,2014Q1,All bedrooms,All property types,Cavan,Euro,441.91


In [3]:
rtb.info()
print("\nNull VALUE share:", f"{rtb['VALUE'].isna().mean():.1%}")

<class 'pandas.DataFrame'>
RangeIndex: 880404 entries, 0 to 880403
Data columns (total 7 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   STATISTIC Label     880404 non-null  str    
 1   Quarter             880404 non-null  str    
 2   Number of Bedrooms  880404 non-null  str    
 3   Property Type       880404 non-null  str    
 4   Location            880404 non-null  str    
 5   UNIT                880404 non-null  str    
 6   VALUE               238587 non-null  float64
dtypes: float64(1), str(6)
memory usage: 47.0 MB

Null VALUE share: 72.9%


In [4]:
# What values do the dimensions take?
for col in ["STATISTIC Label", "Number of Bedrooms", "Property Type", "UNIT"]:
    print(f"--- {col} ({rtb[col].nunique()} unique) ---")
    print(rtb[col].value_counts(dropna=False).to_string())
    print()

print("Quarter range:", rtb["Quarter"].min(), "to", rtb["Quarter"].max())
print("Locations:", rtb["Location"].nunique())
print("Sample locations:", rtb["Location"].drop_duplicates().head(15).tolist())

--- STATISTIC Label (1 unique) ---
STATISTIC Label
RTB Average Monthly Rent Report    880404

--- Number of Bedrooms (7 unique) ---
Number of Bedrooms
All bedrooms     125772
One bed          125772
Two bed          125772
Three bed        125772
1 to 2 bed       125772
1 to 3 bed       125772
Four plus bed    125772

--- Property Type (6 unique) ---
Property Type
All property types     146734
Detached house         146734
Semi detached house    146734
Terrace house          146734
Apartment              146734
Other flats            146734

--- UNIT (1 unique) ---
UNIT
Euro    880404



Quarter range: 2014Q1 to 2025Q3
Locations: 446
Sample locations: ['Carlow', 'Carlow Town', 'Graiguecullen, Carlow', 'Tullow, Carlow', 'Cavan', 'Cavan Town', 'Bailieborough, Cavan', 'Ballyconnell, Cavan', 'Ballyjamesduff, Cavan', 'Belturbet, Cavan', 'Cootehill, Cavan', 'Kingscourt, Cavan', 'Virginia, Cavan', 'Clare', 'Ennis, Clare']


In [5]:
# Where is VALUE actually populated? Null rate by bedroom aggregate vs detail.
(
    rtb.assign(has_value=rtb["VALUE"].notna())
    .groupby("Number of Bedrooms")["has_value"]
    .mean()
    .sort_values(ascending=False)
)

Number of Bedrooms
All bedrooms     0.461104
1 to 3 bed       0.416365
1 to 2 bed       0.284968
Three bed        0.240952
Two bed          0.237970
One bed          0.128407
Four plus bed    0.127214
Name: has_value, dtype: float64

## 2. CSO Annual Earnings by NUTS 3 Region

Small file (~756 rows). Earnings by region, used later as the denominator in the affordability ratio. No county-level breakdown exists — counties get mapped to their parent NUTS 3 region in `02_clean`.

In [6]:
earnings = pd.read_csv(earnings_path)
print(earnings.shape)
earnings.head()

(756, 6)


,Statistic Label,Year,Sex,NUTS 3 Regions,UNIT,VALUE
0,Mean Annual Earnings,2011,Both sexes,State,Euro,39594
1,Mean Annual Earnings,2011,Both sexes,Border,Euro,33572
2,Mean Annual Earnings,2011,Both sexes,Midland,Euro,36176
3,Mean Annual Earnings,2011,Both sexes,West,Euro,36274
4,Mean Annual Earnings,2011,Both sexes,Dublin,Euro,44243


In [7]:
for col in ["Statistic Label", "Sex", "NUTS 3 Regions", "UNIT"]:
    print(f"--- {col} ---")
    print(earnings[col].value_counts(dropna=False).to_string())
    print()

print("Year range:", earnings["Year"].min(), "to", earnings["Year"].max())
print("Null VALUE share:", f"{earnings['VALUE'].isna().mean():.1%}")

--- Statistic Label ---
Statistic Label
Mean Annual Earnings      378
Median Annual Earnings    378

--- Sex ---
Sex
Both sexes    252
Male          252
Female        252

--- NUTS 3 Regions ---
NUTS 3 Regions
State         84
Border        84
Midland       84
West          84
Dublin        84
Mid-East      84
Mid-West      84
South-East    84
South-West    84

--- UNIT ---
UNIT
Euro    756

Year range: 2011 to 2024
Null VALUE share: 0.0%


## 3. Residential Property Price Register

The tricky one. **cp1252** encoding (not UTF-8) because of fadas and the Euro sign. Price is a string with `€` and thousands commas. Eircode ~70% null, Property Size Description ~93% null. Loaded raw here; parsing the price to a number happens in cleaning.

In [8]:
ppr = pd.read_csv(ppr_path, encoding="cp1252")
print(ppr.shape)
print("Columns:", ppr.columns.tolist())
ppr.head()

(782596, 9)
Columns: ['Date of Sale (dd/mm/yyyy)', 'Address', 'County', 'Eircode', 'Price (€)', 'Not Full Market Price', 'VAT Exclusive', 'Description of Property', 'Property Size Description']


C:\Users\kosis\AppData\Local\Temp\ipykernel_17544\202288077.py:1: DtypeWarning: Columns (0: Property Size Description) have mixed types. Specify dtype option on import or set low_memory=False.
  ppr = pd.read_csv(ppr_path, encoding="cp1252")


,Date of Sale (dd/mm/yyyy),Address,County,Eircode,Price (€),Not Full Market Price,VAT Exclusive,Description of Property,Property Size Description
0,01/01/2010,"5 Braemor Drive, Churchtown, Co.Dublin",Dublin,NaN,"€343,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN
1,03/01/2010,"134 Ashewood Walk, Summerhill Lane, Portlaoise",Laois,NaN,"€185,000.00",No,Yes,New Dwelling house /Apartment,greater than or equal to 38 sq metres and less...
2,04/01/2010,"1 Meadow Avenue, Dundrum, Dublin 14",Dublin,NaN,"€438,500.00",No,No,Second-Hand Dwelling house /Apartment,NaN
3,04/01/2010,"1 The Haven, Mornington",Meath,NaN,"€400,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN
4,04/01/2010,"11 Melville Heights, Kilkenny",Kilkenny,NaN,"€160,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN


In [9]:
# Null rates per column
(ppr.isna().mean().sort_values(ascending=False) * 100).round(1).astype(str) + "%"

Property Size Description    93.2%
Eircode                      70.4%
Date of Sale (dd/mm/yyyy)     0.0%
County                        0.0%
Address                       0.0%
Price (€)                     0.0%
Not Full Market Price         0.0%
VAT Exclusive                 0.0%
Description of Property       0.0%
dtype: str

In [10]:
print("--- County (", ppr["County"].nunique(), "unique) ---")
print(ppr["County"].value_counts().to_string())
print("\n--- Not Full Market Price ---")
print(ppr["Not Full Market Price"].value_counts(dropna=False).to_string())
print("\n--- Description of Property ---")
print(ppr["Description of Property"].value_counts(dropna=False).to_string())

--- County ( 26 unique) ---
County
Dublin       245080
Cork          86570
Kildare       42482
Galway        37825
Meath         32569
Limerick      28773
Wexford       27340
Wicklow       26455
Louth         21826
Waterford     21415
Kerry         21367
Tipperary     20958
Donegal       20773
Mayo          18679
Clare         17523
Westmeath     15121
Laois         12993
Kilkenny      12506
Cavan         11609
Sligo         11425
Roscommon     11058
Offaly         9977
Carlow         8567
Leitrim        6904
Longford       6639
Monaghan       6162

--- Not Full Market Price ---
Not Full Market Price
No     742711
Yes     39885

--- Description of Property ---
Description of Property
Second-Hand Dwelling house /Apartment    644102
New Dwelling house /Apartment            138446
Teach/Árasán Cónaithe Atháimhe               44
Teach/Árasán Cónaithe Nua                     3
Teach/?ras?n C?naithe Nua                     1


In [11]:
# Date span + a peek at the raw price strings (note the leading euro sign + commas)
dates = pd.to_datetime(ppr["Date of Sale (dd/mm/yyyy)"], format="%d/%m/%Y")
print("Date of Sale range:", dates.min().date(), "to", dates.max().date())
price_col = [c for c in ppr.columns if c.startswith("Price")][0]
print("Price column name:", repr(price_col))
ppr[price_col].head(5).tolist()

Date of Sale range: 2010-01-01 to 2026-04-24
Price column name: 'Price (€)'


['€343,000.00', '€185,000.00', '€438,500.00', '€400,000.00', '€160,000.00']

## Takeaways for cleaning (`02_clean.ipynb`)

_Fill in after running the cells above — confirm each against the printed output:_

- **RTB:** keep `Quarter` parseable to a period; decide which `Number of Bedrooms` / `Property Type` aggregates to keep vs. drop; the high null `VALUE` is expected and should be dropped or kept depending on the grain we analyse at.
- **Earnings:** filter to a single `Sex` (likely "Both sexes") and one `Statistic Label` (Mean vs Median) for the affordability ratio; 2024 is the latest year, so 2025 rents will be divided by 2024 earnings.
- **PPR:** parse `Price` (strip € and commas → float), parse the date, filter `Not Full Market Price == "No"` for market-rate analysis, and build the county → NUTS 3 region mapping to join with earnings.